In [ ]:
# CONFIGURACIÓN Y PREPARACIÓN DE DATOS

import pandas as pd

# Cargar archivo
df = pd.read_excel("plan_de_compras_2025.xlsx")

# Normalizar encabezados
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
)

# Crear base analítica evitando repetición por detalle de OC
columnas_sin_detalle_oc = [
    columna for columna in df.columns
    if columna not in [
        "cantidad oc asociadas ítem 2025",
        "oc asociada item 2025"
    ]
]

df_analisis = df.drop_duplicates(
    subset=columnas_sin_detalle_oc
).copy()

print("Base preparada correctamente.")
print(f"Registros originales: {len(df)}")
print(f"Registros para análisis: {len(df_analisis)}")

In [ ]:
# 1. RESUMEN EJECUTIVO GENERAL

proyectos_df = (
    df_analisis
    .groupby("id proyecto", as_index=False)
    .agg(
        monto_proyecto=("monto total ítem año 2025", "sum"),
        unidad_compra=("unidad de compra", "first"),
        tipo_proyecto=("tipo proyecto", "first"),
        estado_proyecto=("estado proyecto", "first"),
        tiene_arrastre=("item de arraste", "max")
    )
)

# CALCULAR INDICADORES GENERALES

resumen_general = pd.DataFrame({
    "Indicador": [
        "Proyectos únicos",
        "Unidades de compra",
        "Tipos de proyecto",
        "Estados de proyecto",
        "Monto total planificado",
        "Monto promedio por proyecto",
        "Mediana del monto por proyecto",
        "Proyectos con arrastre"
    ],

    "Resultado": [
        proyectos_df["id proyecto"].nunique(),
        proyectos_df["unidad_compra"].nunique(),
        proyectos_df["tipo_proyecto"].nunique(),
        proyectos_df["estado_proyecto"].nunique(),
        proyectos_df["monto_proyecto"].sum(),
        proyectos_df["monto_proyecto"].mean(),
        proyectos_df["monto_proyecto"].median(),
        (proyectos_df["tiene_arrastre"] == 1).sum()
    ]
})

# FUNCIÓN PARA FORMATO EN PESOS CHILENOS

def formato_clp(valor):
    return f"CLP $ {valor:,.0f}".replace(",", ".")

# CREAR COPIA SOLO PARA PRESENTACIÓN

resumen_general_mostrar = resumen_general.copy()

# Permitir que la columna Resultado pueda contener
# números y texto formateado
resumen_general_mostrar["Resultado"] = (
    resumen_general_mostrar["Resultado"]
    .astype(object)
)

# IDENTIFICAR INDICADORES MONETARIOS

indicadores_monetarios = [
    "Monto total planificado",
    "Monto promedio por proyecto",
    "Mediana del monto por proyecto"
]

# APLICAR FORMATO CLP

mascara_monetaria = (
    resumen_general_mostrar["Indicador"]
    .isin(indicadores_monetarios)
)

resumen_general_mostrar.loc[
    mascara_monetaria,
    "Resultado"
] = (
    resumen_general_mostrar.loc[
        mascara_monetaria,
        "Resultado"
    ]
    .apply(formato_clp)
)

# FORMATEAR LOS INDICADORES NO MONETARIOS

resumen_general_mostrar.loc[
    ~mascara_monetaria,
    "Resultado"
] = (
    resumen_general_mostrar.loc[
        ~mascara_monetaria,
        "Resultado"
    ]
    .apply(lambda x: f"{x:,.0f}".replace(",", "."))
)

# FORMATO VISUAL

resumen_general_estilo = (
    resumen_general_mostrar.style

    .set_properties(**{
        "text-align": "center"
    })

    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center")
            ]
        }
    ])
)

# MOSTRAR TABLA

display(resumen_general_estilo)

In [ ]:
# 2. RESUMEN POR UNIDAD DE COMPRA

# AGRUPAR INFORMACIÓN POR UNIDAD DE COMPRA

resumen_unidad = (
    proyectos_df
    .groupby(
        "unidad_compra",
        dropna=False
    )
    .agg(
        Proyectos=("id proyecto", "nunique"),
        Monto_Total=("monto_proyecto", "sum"),
        Monto_Promedio=("monto_proyecto", "mean")
    )
    .reset_index()
)

# CALCULAR PARTICIPACIÓN SOBRE EL MONTO TOTAL

monto_total_general = proyectos_df["monto_proyecto"].sum()

resumen_unidad["Participacion"] = (
    resumen_unidad["Monto_Total"]
    / monto_total_general
    * 100
)

# ORDENAR DE MAYOR A MENOR MONTO

resumen_unidad = resumen_unidad.sort_values(
    "Monto_Total",
    ascending=False
).reset_index(drop=True)

# CAMBIAR NOMBRES PARA PRESENTACIÓN

resumen_unidad = resumen_unidad.rename(
    columns={
        "unidad_compra": "Unidad de Compra",
        "Monto_Total": "Monto Total (CLP)",
        "Monto_Promedio": "Monto Promedio (CLP)",
        "Participacion": "Participación (%)"
    }
)
fila_total = pd.DataFrame({
    "Unidad de Compra": ["TOTAL GENERAL"],
    "Proyectos": [
        proyectos_df["id proyecto"].nunique()
    ],
    "Monto Total (CLP)": [
        monto_total_general
    ],
    "Monto Promedio (CLP)": [
        proyectos_df["monto_proyecto"].mean()
    ],
    "Participación (%)": [
        100.00
    ]
})

resumen_unidad_mostrar = pd.concat(
    [
        resumen_unidad,
        fila_total
    ],
    ignore_index=True
)

# FORMATO VISUAL

resumen_unidad_estilo = (
    resumen_unidad_mostrar.style

    .format({
        "Proyectos": "{:.0f}",
        "Monto Total (CLP)": formato_clp,
        "Monto Promedio (CLP)": formato_clp,
        "Participación (%)": lambda x: f"{x:.2f}%"
    })

    .set_properties(**{
        "text-align": "center"
    })

    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center")
            ]
        }
    ])

    # Destacar fila TOTAL
    .set_properties(
        subset=pd.IndexSlice[
            [len(resumen_unidad_mostrar) - 1], :
        ],
        **{
            "font-weight": "bold"
        }
    )
)

# MOSTRAR TABLA

display(resumen_unidad_estilo)

### Interpretación

Se observa una alta concentración del presupuesto: la principal Unidad de Compra representa el
87,74 % del monto planificado, y las dos primeras acumulan aproximadamente el 94,69 %.

Aunque varias unidades presentan pocos proyectos, algunas registran montos promedio elevados,
evidenciando diferencias importantes en la magnitud económica de las iniciativas.

In [ ]:
# 3. RESUMEN POR ESTADO DEL PROYECTO

resumen_estado = (
    proyectos_df
    .groupby(
        "estado_proyecto",
        dropna=False
    )
    .agg(
        Proyectos=("id proyecto", "nunique"),
        Monto_Total=("monto_proyecto", "sum"),
        Monto_Promedio=("monto_proyecto", "mean")
    )
    .reset_index()
)

# Calcular participación sobre el monto total general

monto_total_general = proyectos_df["monto_proyecto"].sum()

resumen_estado["Participacion"] = (
    resumen_estado["Monto_Total"]
    / monto_total_general
    * 100
)

# Ordenar de mayor a menor monto

resumen_estado = resumen_estado.sort_values(
    "Monto_Total",
    ascending=False
).reset_index(drop=True)

# Renombrar columnas para presentación

resumen_estado = resumen_estado.rename(
    columns={
        "estado_proyecto": "Estado del Proyecto",
        "Monto_Total": "Monto Total (CLP)",
        "Monto_Promedio": "Monto Promedio (CLP)",
        "Participacion": "Participación (%)"
    }
)
fila_total_estado = pd.DataFrame({
    "Estado del Proyecto": ["TOTAL GENERAL"],
    "Proyectos": [
        proyectos_df["id proyecto"].nunique()
    ],
    "Monto Total (CLP)": [
        monto_total_general
    ],
    "Monto Promedio (CLP)": [
        proyectos_df["monto_proyecto"].mean()
    ],
    "Participación (%)": [
        100.00
    ]
})

# Unir tabla y fila total

resumen_estado_mostrar = pd.concat(
    [
        resumen_estado,
        fila_total_estado
    ],
    ignore_index=True
)

# Formato visual

resumen_estado_estilo = (
    resumen_estado_mostrar.style

    .format({
        "Proyectos": "{:.0f}",
        "Monto Total (CLP)": formato_clp,
        "Monto Promedio (CLP)": formato_clp,
        "Participación (%)": lambda x: f"{x:.2f}%"
    })

    .set_properties(**{
        "text-align": "center"
    })

    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center")
            ]
        }
    ])

    .set_properties(
        subset=pd.IndexSlice[
            [len(resumen_estado_mostrar) - 1], :
        ],
        **{
            "font-weight": "bold"
        }
    )
)

display(resumen_estado_estilo)

### Interpretación

Los proyectos en estado **Actualizado** concentran el **71,83 % del monto planificado**, mientras que los proyectos **Publicados** representan el **28,17 %**.

Aunque los proyectos Publicados son menos numerosos, presentan un monto promedio y una mediana superiores, evidenciando una mayor magnitud económica individual dentro de este grupo.

### Interpretación

El estado **Actualizado** concentra 248 proyectos y el 71,83 % del monto planificado.

Aunque los proyectos **Publicados** son menos numerosos, presentan un monto promedio superior,
lo que evidencia una mayor magnitud económica individual en este grupo.

In [ ]:
# 3.1 COMPARACIÓN DE MONTOS POR ESTADO

comparacion_estado = (
    proyectos_df
    .groupby(
        "estado_proyecto",
        dropna=False
    )
    .agg(
        Proyectos=("id proyecto", "nunique"),
        Monto_Total=("monto_proyecto", "sum"),
        Monto_Promedio=("monto_proyecto", "mean"),
        Monto_Mediana=("monto_proyecto", "median"),
        Monto_Maximo=("monto_proyecto", "max")
    )
    .reset_index()
)

comparacion_estado = comparacion_estado.rename(
    columns={
        "estado_proyecto": "Estado del Proyecto",
        "Monto_Total": "Monto Total (CLP)",
        "Monto_Promedio": "Monto Promedio (CLP)",
        "Monto_Mediana": "Mediana (CLP)",
        "Monto_Maximo": "Monto Máximo (CLP)"
    }
)

comparacion_estado_estilo = (
    comparacion_estado.style

    .format({
        "Proyectos": "{:.0f}",
        "Monto Total (CLP)": formato_clp,
        "Monto Promedio (CLP)": formato_clp,
        "Mediana (CLP)": formato_clp,
        "Monto Máximo (CLP)": formato_clp
    })

    .set_properties(**{
        "text-align": "center"
    })

    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center")
            ]
        }
    ])
)

display(comparacion_estado_estilo)

In [ ]:
# 3.2 ZOOM - PROYECTOS PUBLICADOS POR UNIDAD DE COMPRA

proyectos_publicados = proyectos_df[
    proyectos_df["estado_proyecto"].eq("Publicado")
].copy()

resumen_publicados = (
    proyectos_publicados
    .groupby(
        "unidad_compra",
        dropna=False
    )
    .agg(
        Proyectos=("id proyecto", "nunique"),
        Monto_Total=("monto_proyecto", "sum"),
        Monto_Promedio=("monto_proyecto", "mean"),
        Mediana=("monto_proyecto", "median")
    )
    .reset_index()
)

monto_total_publicados = proyectos_publicados[
    "monto_proyecto"
].sum()

resumen_publicados["Participacion"] = (
    resumen_publicados["Monto_Total"]
    / monto_total_publicados
    * 100
)

resumen_publicados = resumen_publicados.sort_values(
    "Monto_Total",
    ascending=False
).reset_index(drop=True)

resumen_publicados = resumen_publicados.rename(
    columns={
        "unidad_compra": "Unidad de Compra",
        "Monto_Total": "Monto Total (CLP)",
        "Monto_Promedio": "Monto Promedio (CLP)",
        "Mediana": "Mediana (CLP)",
        "Participacion": "Participación Publicados (%)"
    }
)

resumen_publicados_estilo = (
    resumen_publicados.style

    .format({
        "Proyectos": "{:.0f}",
        "Monto Total (CLP)": formato_clp,
        "Monto Promedio (CLP)": formato_clp,
        "Mediana (CLP)": formato_clp,
        "Participación Publicados (%)":
            lambda x: f"{x:.2f}%"
    })

    .set_properties(**{
        "text-align": "center"
    })

    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center")
            ]
        }
    ])
)

display(resumen_publicados_estilo)

In [ ]:
# 3.3 TOP 10 PROYECTOS PUBLICADOS POR MONTO

top10_publicados = (
    proyectos_publicados[
        [
            "id proyecto",
            "unidad_compra",
            "monto_proyecto"
        ]
    ]
    .sort_values(
        "monto_proyecto",
        ascending=False
    )
    .head(10)
    .copy()
)

# Participación de cada proyecto sobre el total Publicado
top10_publicados["Participacion"] = (
    top10_publicados["monto_proyecto"]
    / monto_total_publicados
    * 100
)

# Participación acumulada
top10_publicados["Participacion_Acumulada"] = (
    top10_publicados["Participacion"]
    .cumsum()
)

# Renombrar para presentación
top10_publicados = top10_publicados.rename(
    columns={
        "id proyecto": "ID Proyecto",
        "unidad_compra": "Unidad de Compra",
        "monto_proyecto": "Monto Proyecto (CLP)",
        "Participacion": "Participación (%)",
        "Participacion_Acumulada": "Participación Acumulada (%)"
    }
)

top10_publicados_estilo = (
    top10_publicados.style

    .format({
        "Monto Proyecto (CLP)": formato_clp,
        "Participación (%)": lambda x: f"{x:.2f}%",
        "Participación Acumulada (%)": lambda x: f"{x:.2f}%"
    })

    .set_properties(**{
        "text-align": "center"
    })

    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center")
            ]
        }
    ])
)

display(top10_publicados_estilo)

### Interpretación ejecutiva — Proyectos Publicados

Aunque los proyectos en estado **Publicado** representan solo 78 de los 326 proyectos analizados,presentan un monto promedio de **CLP $60,5 millones**, superior a los **CLP $48,5 millones**
de los proyectos Actualizados.

La diferencia también se observa en la mediana: los proyectos Publicados alcanzan aproximadamente **CLP $12,6 millones**, frente a **CLP $3,0 millones** en los Actualizados. Esto indica que el mayor promedio de los proyectos Publicados no se explica únicamente por valores extremos, sino también por una mayor magnitud económica de sus proyectos en términos generales.

Dentro de los proyectos Publicados existe además una fuerte concentración presupuestaria: las dos principales Unidades de Compra reúnen aproximadamente el **81,15 % del monto publicado**.
La principal concentra por sí sola el **56,49 %**, con un monto promedio cercano a **CLP $140,3 millones por proyecto**.

En conjunto, los resultados muestran que el estado Publicado agrupa menos proyectos, pero con un mayor peso económico individual y una alta concentración del presupuesto en determinadas Unidades de Compra.

In [ ]:
# 4. RESUMEN POR TIPO DE PROYECTO

resumen_tipo = (
    proyectos_df
    .groupby(
        "tipo_proyecto",
        dropna=False
    )
    .agg(
        Proyectos=("id proyecto", "nunique"),
        Monto_Total=("monto_proyecto", "sum"),
        Monto_Promedio=("monto_proyecto", "mean")
    )
    .reset_index()
)

# Calcular participación sobre el monto total general

monto_total_general = proyectos_df["monto_proyecto"].sum()

resumen_tipo["Participacion"] = (
    resumen_tipo["Monto_Total"]
    / monto_total_general
    * 100
)

# Ordenar de mayor a menor monto

resumen_tipo = resumen_tipo.sort_values(
    "Monto_Total",
    ascending=False
).reset_index(drop=True)

# Renombrar columnas

resumen_tipo = resumen_tipo.rename(
    columns={
        "tipo_proyecto": "Tipo de Proyecto",
        "Monto_Total": "Monto Total (CLP)",
        "Monto_Promedio": "Monto Promedio (CLP)",
        "Participacion": "Participación (%)"
    }
)

fila_total_tipo = pd.DataFrame({
    "Tipo de Proyecto": ["TOTAL GENERAL"],
    "Proyectos": [
        proyectos_df["id proyecto"].nunique()
    ],
    "Monto Total (CLP)": [
        monto_total_general
    ],
    "Monto Promedio (CLP)": [
        proyectos_df["monto_proyecto"].mean()
    ],
    "Participación (%)": [
        100.00
    ]
})

resumen_tipo_mostrar = pd.concat(
    [
        resumen_tipo,
        fila_total_tipo
    ],
    ignore_index=True
)

resumen_tipo_estilo = (
    resumen_tipo_mostrar.style

    .format({
        "Proyectos": "{:.0f}",
        "Monto Total (CLP)": formato_clp,
        "Monto Promedio (CLP)": formato_clp,
        "Participación (%)": lambda x: f"{x:.2f}%"
    })

    .set_properties(**{
        "text-align": "center"
    })

    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center")
            ]
        }
    ])

    .set_properties(
        subset=pd.IndexSlice[
            [len(resumen_tipo_mostrar) - 1], :
        ],
        **{
            "font-weight": "bold"
        }
    )
)

display(resumen_tipo_estilo)